# Multi-Modal Neural Network for Construction Cost Prediction

This notebook builds a model combining:
- Tabular data (MLP)
- Sentinel-2 imagery (CNN)
- VIIRS imagery (CNN)

We fuse all modalities into a final regression model.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import datetime
import random
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import product
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



print("Using device:", device)

### Random handling

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Dataset handling
Loading and preparing dataset for the model

### Loading Data

In [ ]:
DataPath = Path("..") / "Processed data"
ImgPath = DataPath / "processed_composite"
ModelPath = Path("..") / "Models"

#Loading all images into a dict to avoid loading them multiple times during training
tensor_dict = {}
for img_file in ImgPath.glob("*.pt"):
    img_dict = torch.load(img_file, weights_only=True)
    tensor = torch.cat([img_dict['sentinel'], img_dict['viirs']], dim=0)
    tensor_dict[img_file.name] = tensor

full_df = pd.read_csv(DataPath / "processed_data.csv")
print(f"Full shape: {full_df.shape}")

japan_df = pd.read_csv(DataPath / "processed_japan.csv")
print(f"Japan shape: {japan_df.shape}")

philippines_df = pd.read_csv(DataPath / "processed_philippines.csv")
print(f"Philippines shape: {philippines_df.shape}")

eval_df = pd.read_csv(DataPath / "processed_eval.csv")
print(f"Eval shape: {eval_df.shape}")

full_df.head()

### Possible columns in the data

In [ ]:
numeric_cols = [
    'deflated_gdp_usd',
    'us_cpi',
    'straight_distance_to_capital_km',
    'quarter_label'
]

categorical_cols = [
    "geolocation_name",
    "country",
    "landlocked",
    "region_economic_classification",
    "access_to_airport",
    "access_to_port",
    "access_to_highway",
    "access_to_railway",
    "seismic_hazard_zone",
    "flood_risk_class",
    "tropical_cyclone_wind_risk",
    "tornadoes_wind_risk",
    "koppen_climate_zone",
]

target_col = 'construction_cost_per_m2_usd'

img_col = "processed_imgs"

id_col = "data_id"

### Dataset class
For easier access of the data for the models

In [ ]:
class ConstructionDataset(Dataset):
    """
    Loads one row at a time from the CSV and lazily loads the matching
    .pt tensor file from tensor_dict.

    Args:
        df:             pandas DataFrame (already split into train/val)
        tensor_dict:    directory containing image tensors of size (13, H, W)
        augment:        whether to apply random horizontal/vertical flips
    """

    def __init__(
        self,
        df: pd.DataFrame,
        tensor_dict: dict,
        augment: bool = False,
    ):
        self.df = df.reset_index(drop=True)
        self.tensor_dict = tensor_dict
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        data_id = row[id_col]
        continuous = torch.tensor(row[numeric_cols].values.astype(np.float32), dtype=torch.float32)
        categoricals = {name: torch.tensor(int(row[name]), dtype=torch.long) for name in categorical_cols if name in self.df.columns}
        images = self.tensor_dict[row[img_col]]
        if target_col in self.df.columns:
            target = torch.tensor(row[target_col], dtype=torch.float32)
        else: 
            target = torch.tensor(np.nan, dtype=torch.float32)

        if self.augment:
            if torch.rand(1) > 0.5:
                images = torch.flip(images, dims=[2])    # horizontal flip
            if torch.rand(1) > 0.5:
                images = torch.flip(images, dims=[1])    # vertical flip

        return data_id, continuous, categoricals, images, target

### Interpolation
For temporal interpolation per geolocation

In [ ]:
def interpolate_datapoints(
        a: pd.Series,
        b: pd.Series,
        quarter_label: float,
        tensor_dict: dict,
        numeric_cols: list,
        categorical_cols: list,
        img_col: str,
        target_col: str,
        lam: float
    ) -> pd.Series:
    """
    Interpolate two datapoints and all their values according to their type.
 
    numeric / tensor  →  lam*a + (1-lam)*b
    categorical       →  a if lam >= 0.5 else b   (dominant-sample rule)
    """
    c = pd.Series()
    keys = a.keys()
    numeric_cols = [col for col in numeric_cols if col in keys]
    categorical_cols = [col for col in categorical_cols if col in keys]
    
    for col in numeric_cols:
        if col == 'quarter_label': c[col] = quarter_label
        else: c[col] = lam * a[col] + (1 - lam) * b[col]

    for col in categorical_cols:
        c[col] = a[col] if lam >= 0.5 else b[col]

    c_path = "Interpolation/" + a[img_col][:-3] + '_' + b[img_col][:-3] + '.pt'
    c[img_col] = c_path
    tensor_dict[c_path] = torch.lerp(tensor_dict[b[img_col]], tensor_dict[a[img_col]], lam)

    c[target_col] = lam * a[target_col] + (1 - lam) * b[target_col]
    
    return c, tensor_dict

def geolocation_temporal_interpolation(df: pd.DataFrame, tensor_dict: dict, max_missing: int = None):

    quarter_labels = sorted(df['quarter_label'].unique())
    geolocations = sorted(df['geolocation_name'].unique())

    for geo in geolocations:
        geo_df = df[df['geolocation_name'] == geo]
        geo_quarters = set(geo_df['quarter_label'].values)
        last_quarter = None
        missing_quarters = []
        row = None
        for quarter in quarter_labels:
            if quarter in geo_quarters:
                new_row = geo_df[geo_df['quarter_label'] == quarter].iloc[0]
                interpolations = len(missing_quarters)
                if last_quarter != None and (max_missing == None or interpolations <= max_missing):
                    delta_lambda = 1 / (1 + interpolations)
                    synthetic_rows = []
                    for i in range(interpolations):
                        lam = 1 - delta_lambda * (1 + i)
                        synth_row, synth_tensor_dict = interpolate_datapoints(
                            row,
                            new_row,
                            missing_quarters[i],
                            tensor_dict,
                            numeric_cols,
                            categorical_cols,
                            img_col,
                            target_col,
                            lam
                        )
                        tensor_dict = synth_tensor_dict
                        synthetic_rows.append(synth_row)
                    df_synthetic = pd.DataFrame(synthetic_rows, columns=df.columns)
                    df = pd.concat([df, df_synthetic], ignore_index=True)
                missing_quarters = []
                row = new_row
                last_quarter = quarter
            else:
                if last_quarter != None:
                    missing_quarters.append(quarter)
    
    return df, tensor_dict

#### Visualize interpolation

In [ ]:
import matplotlib.patches as mpatches

def plot_quarter_heatmap(df: pd.DataFrame, max_locations: int = None):

    quarter_labels = sorted(df['quarter_label'].unique())
    
    geolocations = sorted(df['geolocation_name'].unique())
    
    costs = df['construction_cost_per_m2_usd']
    min_cost = costs.min()
    max_cost = costs.max()

    # Matrix of normalized costs, NaN where missing
    matrix = []
    count = 0
    for geo in geolocations:
        if max_locations != None and count >= max_locations: break
        geo_df = df[df['geolocation_name'] == geo]
        geo_quarters = set(geo_df['quarter_label'].values)
        row = []
        for q in quarter_labels:
            if q in geo_quarters:
                cost = geo_df[geo_df['quarter_label'] == q]['construction_cost_per_m2_usd'].values[0]
                norm_cost = (cost - min_cost) / (max_cost - min_cost)  # fix: brackets around numerator
                row.append(norm_cost)
            else:
                row.append(np.nan)  # NaN for missing — drives the masked colormap
        matrix.append(row)
        count += 1
    matrix = np.array(matrix)
    
    # Colormap: green -> yellow -> red for present, gray for missing (NaN)
    cmap = plt.cm.RdYlGn_r                  # red=high, green=low
    cmap.set_bad(color='#e0e0e0')           # gray for NaN cells

    height = len(geolocations) if max_locations == None else max_locations
    fig, ax = plt.subplots(figsize=(len(quarter_labels) * 0.6, height * 0.5 + 1.5))
    
    im = ax.imshow(matrix, cmap=cmap, aspect='auto', vmin=0, vmax=1)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.04)
    cbar.set_ticks([0, 0.5, 1])
    cbar.set_ticklabels([f'${min_cost:,.0f}', f'${(min_cost+max_cost)/2:,.0f}', f'${max_cost:,.0f}'])
    cbar.set_label('Cost per m² (USD)', fontsize=9)

    # Axis labels
    ax.set_xticks(range(len(quarter_labels)))
    ax.set_xticklabels(quarter_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(height))
    ax.set_yticklabels(range(height))
    
    # Legend
    missing_patch = mpatches.Patch(color='#e0e0e0', label='No data')
    ax.legend(handles=[missing_patch], loc='lower right',
              bbox_to_anchor=(1.18, -0.22), frameon=False, fontsize=9)
    
    ax.set_title('Construction cost per m² by quarter', fontsize=13, pad=20, loc='left')
    plt.tight_layout()
    plt.show()

#plot_quarter_heatmap(japan_df, 5)
#new_japan_df, tensor_dict = geolocation_temporal_interpolation(japan_df, tensor_dict)
#plot_quarter_heatmap(new_japan_df, 5)

## Model Definition

### Multimodal Construction Cost Predictor

#### Tabular Encoder
Takes in tabular features and outputs a 128-d tensor

In [ ]:
class TabularEncoder(nn.Module):
    """
    Embeds categorical features + passes continuous features through MLP.
    Output: fixed 128-d vector.
    """

    def __init__(
        self,
        continuous_dim: int,
        categorical_vocab: dict,
        embed_dim_fn=lambda v: min(50, (v + 1) // 2),
        hidden_dim: int = 256,
        out_dim: int = 128,
        dropout: float = 0.3,
    ):
        super().__init__()

        self.embeddings = nn.ModuleDict({
            name: nn.Embedding(vocab_size + 1, embed_dim_fn(vocab_size))
            for name, vocab_size in categorical_vocab.items()
        })

        total_embed_dim = sum(
            embed_dim_fn(v) for v in categorical_vocab.values()
        )
        mlp_in = continuous_dim + total_embed_dim

        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, out_dim),
            nn.ReLU(),
        )

    def forward(self, continuous: torch.Tensor, categoricals: dict) -> torch.Tensor:
        """
        Args:
            continuous:   (B, continuous_dim)  float32, already standardised
            categoricals: dict of {name: (B,) int64 tensors}
        Returns:
            (B, out_dim)
        """
        parts = [continuous]
        for name, emb in self.embeddings.items():
            parts.append(emb(categoricals[name]))
        x = torch.cat(parts, dim=1)
        return self.mlp(x)

#### Lightweight CNN image encoder

In [ ]:
class ResBlock(nn.Module):
    """Simple pre-activation residual block."""

    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
        )

    def forward(self, x):
        return x + self.block(x)

class ImageEncoder(nn.Module):
    """
    Small CNN for 13-channel (Sentinel-2 + VIIRS) imagery at 224x224.
    Projects to a low-dimensional vector (default 64-d) to avoid
    dominating the tabular signal at fusion time.

    Deliberately kept shallow (3 stages) to reduce overfitting on a
    small dataset (1024 samples) and to keep image embedding compact.
    """

    def __init__(
        self,
        in_channels: int = 13, #1 for VIIRS and 12 for Sentinel
        base_ch: int = 32,
        out_dim: int = 64,
        dropout: float = 0.3,
    ):
        super().__init__()

        # Stage 1: 224 -> 56  (4x stride)
        self.stage1 = nn.Sequential(
            nn.Conv2d(in_channels, base_ch, kernel_size=7, stride=4, padding=3, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(),
            ResBlock(base_ch),
        )
        # Stage 2: 56 -> 14  (4x stride via 2 poolings)
        self.stage2 = nn.Sequential(
            nn.Conv2d(base_ch, base_ch * 2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_ch * 2),
            nn.ReLU(),
            ResBlock(base_ch * 2),
            nn.Conv2d(base_ch * 2, base_ch * 2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_ch * 2),
            nn.ReLU(),
            ResBlock(base_ch * 2),
        )
        # Stage 3: 14 -> 7
        self.stage3 = nn.Sequential(
            nn.Conv2d(base_ch * 2, base_ch * 4, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_ch * 4),
            nn.ReLU(),
            ResBlock(base_ch * 4),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)   # -> (B, C, 1, 1)

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(base_ch * 4, out_dim),
            nn.ReLU(),
        )

    def forward(self, imgs: torch.Tensor) -> torch.Tensor:
        """
        Args:
            imgs: (B, 13, H, W)  float32, channel order: sentinel[0:12] + viirs[12]
        Returns:
            (B, out_dim)
        """
        x = self.stage1(imgs)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.pool(x).flatten(1)
        return self.head(x)

#### Gated fusion

In [ ]:
class GatedFusion(nn.Module):
    """
    Learns a per-sample gate vector in [0,1]^d that blends the two
    modality embeddings.  The gate is computed from the concatenation
    of both embeddings, so it can route information context-dependently.

    Output dimension = out_dim (tabular projected up, image projected up/down).
    """

    def __init__(self, tab_dim: int = 128, img_dim: int = 64, out_dim: int = 128):
        super().__init__()
        self.out_dim = out_dim

        # Project both modalities to a common dimension
        self.tab_proj = nn.Linear(tab_dim, out_dim)
        self.img_proj = nn.Linear(img_dim, out_dim)

        # Gate: sigmoid output in (0,1)^out_dim
        self.gate_net = nn.Sequential(
            nn.Linear(tab_dim + img_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
            nn.Sigmoid(),
        )

    def forward(self, tab: torch.Tensor, img: torch.Tensor) -> torch.Tensor:
        """
        Args:
            tab: (B, tab_dim)
            img: (B, img_dim)
        Returns:
            (B, out_dim)
        """
        gate  = self.gate_net(torch.cat([tab, img], dim=1))   # (B, out_dim)
        tab_p = self.tab_proj(tab)                            # (B, out_dim)
        img_p = self.img_proj(img)                            # (B, out_dim)
        return gate * tab_p + (1 - gate) * img_p

#### Full multimodal model

In [ ]:
class ConstructionCostModel(nn.Module):
    """
    End-to-end model:
      tabular features + satellite imagery  ->  construction cost per m2

    Modality balance
    ----------------
    - Tabular encoder produces 128-d, image encoder 64-d.
    - GatedFusion projects both to 128-d with a learned gate, so the
      network can suppress image noise when tabular features are
      sufficient and boost image contribution when helpful.
    - Result: tabular signal has double the raw dimensionality before
      fusion, preventing the high-capacity CNN from dominating.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        tab_out_dim: int        = 128,
        img_out_dim: int        = 64,
        fusion_dim: int         = 128,
        head_hidden: int        = 64,
        dropout: float          = 0.3,
        img_channels: int       = 13,
    ):
        super().__init__()

        cols = df.columns

        continuous_dim = 0
        for num_col in numeric_cols:
            if num_col in cols:
                continuous_dim += 1
        categorical_vocab = {}
        for cat_col in categorical_cols:
            if cat_col in cols:
                categorical_vocab[cat_col] = int(df[cat_col].max())
        
        self.tabular_encoder = TabularEncoder(
            continuous_dim    = continuous_dim,
            categorical_vocab = categorical_vocab,
            out_dim           = tab_out_dim,
            dropout           = dropout,
        )
        self.image_encoder = ImageEncoder(
            in_channels = img_channels,
            out_dim     = img_out_dim,
            dropout     = dropout,
        )
        self.fusion = GatedFusion(
            tab_dim = tab_out_dim,
            img_dim = img_out_dim,
            out_dim = fusion_dim,
        )
        self.regression_head = nn.Sequential(
            nn.Linear(fusion_dim, head_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
        )

    def forward(
        self,
        continuous:   torch.Tensor,   # (B, continuous_dim)
        categoricals: dict,           # {name: (B,) int64}
        images:       torch.Tensor,   # (B, 13, H, W)
    ) -> torch.Tensor:
        """Returns (B,) predicted construction cost."""
        tab   = self.tabular_encoder(continuous, categoricals)
        img   = self.image_encoder(images)
        fused = self.fusion(tab, img)
        return self.regression_head(fused).squeeze(1)

    def parameter_groups(self, lr_tab=1e-3, lr_img=2e-4, lr_head=1e-3):
        """
        Returns parameter groups with separate learning rates.
        The image encoder gets a lower lr to avoid overpowering the
        tabular path early in training.
        """
        return [
            {"params": self.tabular_encoder.parameters(), "lr": lr_tab},
            {"params": self.image_encoder.parameters(),   "lr": lr_img},
            {"params": list(self.fusion.parameters()) +
                       list(self.regression_head.parameters()), "lr": lr_head},
        ]

## Training

### Loss Function

In [ ]:
def RMSLELoss(preds, targets):
    preds = torch.clamp(preds, min=0)
    log_diff = torch.log1p(preds) - torch.log1p(targets)
    return torch.sqrt(torch.mean(log_diff ** 2))

### Collate Function

In [ ]:
from torch.utils.data.dataloader import default_collate

def collate_fn(batch):
    data_ids    = [item[0] for item in batch]
    continuous  = default_collate([item[1] for item in batch])
    categoricals = default_collate([item[2] for item in batch])
    images      = default_collate([item[3] for item in batch])
    targets     = default_collate([item[4] for item in batch])
    return data_ids, continuous, categoricals, images, targets

### Training functions

In [ ]:
def train_model(
        model : torch.nn.Module,
        optimizer : torch.optim.Optimizer,
        scheduler : torch.optim.lr_scheduler,
        train_loader : DataLoader,
        val_loader : DataLoader,
        epochs : int
    ):
    train_losses = []
    val_losses = []
    best_model_state = None
    best_val_loss = float('inf')

    print(f'{datetime.datetime.now().time()}  |  Starting training...')

    for epoch in range(1, epochs + 1):

        #Training
        model.train()
        total_loss = 0.0
        for _, continuous, categoricals, images, target in train_loader:
            continuous = continuous.to(device)
            categoricals = {k: v.to(device) for k, v in categoricals.items()}
            images = images.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            pred = model(continuous, categoricals, images)
            loss = RMSLELoss(pred, target)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(target)

        avg_train_loss = total_loss / len(train_loader.dataset)

        train_losses.append(avg_train_loss)
        
        #Validation
        
        model.eval()
        val_loss = 0.0
        for _, continuous, categoricals, images, target in val_loader:
            with torch.no_grad():
                continuous = continuous.to(device)
                categoricals = {k: v.to(device) for k, v in categoricals.items()}
                images = images.to(device)
                target = target.to(device)
                pred = model(continuous, categoricals, images)
                loss = RMSLELoss(pred, target)
                val_loss += loss.item() * len(target)
                
        avg_val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()

        if scheduler is not None: scheduler.step(avg_val_loss)

        if epoch in (1, epochs) or epoch % 10 == 0:
            print('{}  |  Epoch {}  |  Training loss {:.3f} |  Validation loss {:.3f}'.format(datetime.datetime.now().time(), epoch, avg_train_loss, avg_val_loss))
    
    return train_losses, val_losses, best_model_state, best_val_loss

def train_on_df(df, modelName, param_grid):

    best_model = None
    best_loss = float('inf')
    best_train_losses = None
    best_val_losses = None
    best_params = None
    best_train = None
    
    train, val = train_test_split(df, test_size=0.2, random_state=42)
    count = 0
    amount = len(param_grid)
    for params in param_grid:
        nr_epochs, learning_rate, weight_decay, patience, factor, batch_size, dropout, augment, max_missing = params
        count += 1
        
        #TODO: change training data here
        new_train, new_tensor_dict = geolocation_temporal_interpolation(train, tensor_dict, max_missing)

        print(f"Training {modelName} ({count}/{amount}) with shape {new_train.shape}")
        train_dataset = ConstructionDataset(new_train, tensor_dict=new_tensor_dict, augment=augment)
        val_dataset = ConstructionDataset(val, tensor_dict=new_tensor_dict)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

        model = ConstructionCostModel(df=df, dropout=dropout).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=patience, factor=factor)

        train_losses, val_losses, best_model_state, best_val_loss = train_model(model, optimizer, scheduler, train_loader, val_loader, epochs=nr_epochs)
        
        if best_val_loss <= best_loss:
            best_model = best_model_state
            best_loss = best_val_loss
            best_params = params
            best_train_losses = train_losses
            best_val_losses = val_losses
            best_train = train

    torch.save({
        'model': best_model,
        'best_loss': best_loss,
        'best_params' : best_params,
        'train_losses': best_train_losses,
        'val_losses': best_val_losses,
        'training_data': best_train,
        'validation_data': val,
    }, ModelPath / f"{modelName}.pth")

### Training the models
Training with hyperparameter tuning

In [ ]:
#Hyperparameters to test

nr_epochs = [50] #Number of epochs to train for

#Data
augment = [True, False]
max_missing = [None, 0, 1, 3]

#Optimizer
learning_rates = [3e-3] #Starting learning rates
weight_decays = [1e-4, 1e-3] #L2 regularization

#Scheduler
patiences = [10] #Epochs to wait before decreasing the learning rate
factors = [0.1, 0.5] #Factor by which the learning rate will be reduced. new_lr = lr * factor

#Model training
batch_sizes = [128] #Batch sizes to try during training
dropouts = [0.3, 0.5] #Dropout rate in the tabular model

param_grid = list(product(nr_epochs, learning_rates, weight_decays, patiences, factors, batch_sizes, dropouts, augment, max_missing))

#Uncomment the following lines to retrain the models. Othersise, the pre-trained models will be loaded from the Models directory.
train_on_df(full_df, "full_model", param_grid)
train_on_df(japan_df, "japan_model", param_grid)
train_on_df(philippines_df, "philippines_model", param_grid)

## Evaluation
Evaluating which solution is better, full model or split model.

In [ ]:
def evaluate_model(modelName):
    checkpoint = torch.load(ModelPath / f"{modelName}.pth", weights_only=False)
    best_loss = checkpoint['best_loss']
    params = checkpoint['best_params']
    validation_losses = checkpoint['val_losses']
    print(f"{modelName}, Best loss: {best_loss}, Params: {params}")
    epoch = 1
    for loss in validation_losses:
        if loss == best_loss: 
            print(f"Best performance achieved in epoch: {epoch}")
            break
        epoch += 1
    return best_loss
        

loss_full = evaluate_model("full_model")
loss_jp = evaluate_model("japan_model")
loss_ph = evaluate_model("philippines_model")

#Calculating the combined performance of japan_model and philippines_model
len_jp = len(japan_df)
len_ph = len(philippines_df)
len_total = len_jp + len_ph
combined_loss = (len_jp / len_total) * loss_jp + (len_ph / len_total) * loss_ph
print(f"Combined loss for jp and ph: {combined_loss}")

### Plots
Plotting training and validation loss during training for each model

In [ ]:
def plot_losses(modelName):
    checkpoint = torch.load(ModelPath / f"{modelName}.pth", weights_only=False)
    train_losses = checkpoint['train_losses']
    val_losses = checkpoint['val_losses']
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title(f'Training and Validation Loss for {modelName}')
    plt.xlabel('Epochs')
    plt.ylabel('RMSLE Loss')
    plt.legend()
    plt.grid()
    plt.show()


plot_losses("full_model")
plot_losses("japan_model")
plot_losses("philippines_model")

### Submission
Make a submission file for each of the model solutions

In [ ]:
submissionPath = Path("..") / "Submissions"
split_submission = pd.DataFrame()
full_submission = pd.DataFrame()
data_ids = []
split_predictions = []
full_predictions = []

#Converting data to the dataset the model will accept
eval_dataset = ConstructionDataset(eval_df, tensor_dict)

#Loading each model
full_checkpoint = torch.load(ModelPath / "full_model.pth", weights_only=False)
full_model = ConstructionCostModel(full_df, dropout=0.0)
full_model.load_state_dict(full_checkpoint['model'])
full_model.to(device)
full_model.eval()

philippines_checkpoint = torch.load(ModelPath / "philippines_model.pth", weights_only=False)
philippines_model = ConstructionCostModel(philippines_df, dropout=0.0)
philippines_model.load_state_dict(philippines_checkpoint['model'])
philippines_model.to(device)
philippines_model.eval()

japan_checkpoint = torch.load(ModelPath / "japan_model.pth", weights_only=False)
japan_model = ConstructionCostModel(japan_df, dropout=0.0)
japan_model.load_state_dict(japan_checkpoint['model'])
japan_model.to(device)
japan_model.eval()

#Predicting
for data_id, continuous, categoricals, images, target in eval_dataset:
    data_ids.append(data_id)
    country = categoricals['country'].item()
    
    with torch.no_grad():
        cont = continuous.unsqueeze(0).to(device)
        cats = {k: v.unsqueeze(0).to(device) for k, v in categoricals.items()}
        imgs = images.unsqueeze(0).to(device)
        if country == 0: pred = philippines_model(cont, cats, imgs)
        else: pred = japan_model(cont, cats, imgs)

        full_pred = full_model(cont, cats, imgs)
        
    full_predictions.append(full_pred.item())
    split_predictions.append(pred.item())

#Creating submission files
split_submission["data_id"] = data_ids
split_submission["construction_cost_per_m2_usd"] = split_predictions
split_submission.to_csv(submissionPath / "split_submission.csv", index=False)

#full_submission["data_id"] = data_ids
#full_submission["construction_cost_per_m2_usd"] = full_predictions
#full_submission.to_csv(submissionPath / "full_submission.csv", index=False)